# SCD1 Source to Bronze — Auto Loader

This notebook uses **Auto Loader** (`cloudFiles`) to incrementally read CSV files from the input volume, infer the schema, write the data into the Bronze Delta table `data_modeliing_demo.bronze.scd1`, and validate the data quality.

In [0]:
# Auto Loader: Read CSV from input path and write to Bronze Delta table
from pyspark.sql.functions import *

input_path = "/Volumes/data_modeliing_demo/source/demo_volume/scd1/input/"
checkpoint_path = "/Volumes/data_modeliing_demo/source/demo_volume/scd1/checkpoint/"
target_table = "data_modeliing_demo.bronze.scd1"

print(f"Input path    : {input_path}")
print(f"Checkpoint    : {checkpoint_path}")
print(f"Target table  : {target_table}")
print("-" * 80)

# Build the Auto Loader read stream
autoloader_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(input_path)
)

print("Auto Loader stream configured. Starting batch write to Bronze table...")

# Write stream to the Delta table with trigger(availableNow=True) for batch-mode processing
query = (autoloader_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

print(f"\n✅ Data written to {target_table} successfully.")

In [0]:
# Read the Bronze table and display a sample
raw_df = spark.table("data_modeliing_demo.bronze.scd1")

print(f"Total rows loaded: {raw_df.count()}")
print(f"Total columns: {len(raw_df.columns)}")
print("\nSchema:")
raw_df.printSchema()

In [0]:
# Display the raw data sample
display(raw_df)